# Analyzing word and document frequency: tf-idf

A central question in text mining and natural language processing is how to quantify what a document is about. Can we do this by looking at the words that make up the document?

One measure of how important a word may be is its term frequency (tf).

This is how frequently a word occurs in a document - as we saw in Lab 2. However, there are words in a document that occur many times but may not be important. In English these words are most often things like “the”, “is”, “of”, and so forth. We might take the approach of adding words like these to a list of stop words and removing them before analysis, but it is possible that some of these words might be more important in some documents than others. A list of stop words is not a very sophisticated approach to adjusting term frequency for commonly used words.

Another approach is to look at a term’s inverse document frequency (idf), which decreases the weight for commonly used words and increases the weight for words that are not used very much in a collection of documents. This can be combined with term frequency to calculate a term’s tf-idf (the two quantities multiplied together), the frequency of a term adjusted for how rarely it is used.

The tf-idf statistic is intended to measure how important a word is to a document in a collection (or corpus) of documents, for example, to one novel in a collection of novels or to one website in a collection of websites.

### Preparing data

In [1]:
import requests
import string
import pandas as pd

# Jane Eyre
book_url = 'https://www.gutenberg.org/files/1260/1260-0.txt'
response = requests.get(book_url)
bronte1 = response.text
allowed_chars = string.ascii_letters + string.digits + string.whitespace
bronte1 = ''.join(c for c in bronte1 if c in allowed_chars)

# Wuthering Heights
book_url = 'https://www.gutenberg.org/cache/epub/768/pg768.txt'
response = requests.get(book_url)
bronte2 = response.text
allowed_chars = string.ascii_letters + string.digits + string.whitespace
bronte2 = ''.join(c for c in bronte2 if c in allowed_chars)

# Vilette
book_url = 'https://www.gutenberg.org/files/9182/9182-0.txt'
response = requests.get(book_url)
bronte3 = response.text
allowed_chars = string.ascii_letters + string.digits + string.whitespace
bronte3 = ''.join(c for c in bronte3 if c in allowed_chars)

# Agnes Gray
book_url = 'https://www.gutenberg.org/files/767/767-0.txt'
response = requests.get(book_url)
bronte4 = response.text
allowed_chars = string.ascii_letters + string.digits + string.whitespace
bronte4 = ''.join(c for c in bronte4 if c in allowed_chars)

# Create our dataframes
bronte1_lines = bronte1.splitlines()

bronte1_df = pd.DataFrame({
    "line": bronte1_lines,
    "line_number": list(range(len(bronte1_lines)))
})

bronte2_lines = bronte2.splitlines()

bronte2_df = pd.DataFrame({
    "line": bronte2_lines,
    "line_number": list(range(len(bronte2_lines)))
})

bronte3_lines = bronte3.splitlines()

bronte3_df = pd.DataFrame({
    "line": bronte3_lines,
    "line_number": list(range(len(bronte3_lines)))
})

bronte4_lines = bronte4.splitlines()

bronte4_df = pd.DataFrame({
    "line": bronte4_lines,
    "line_number": list(range(len(bronte4_lines)))
})

# We’ll want to know which content comes from which book
bronte1_df = bronte1_df.assign(book = 'Jane Eyre')
bronte2_df = bronte2_df.assign(book = 'Wuthering Heights')
bronte3_df = bronte3_df.assign(book = 'Vilette')
bronte4_df = bronte4_df.assign(book = 'Agnes Grey')

# Finally, we concatenate the books into one dataframe
books = [bronte1_df, bronte2_df, bronte3_df, bronte4_df]
bronte_books_df = pd.concat(books)
bronte_books_df.head()

,line,line_number,book
0,START OF THE PROJECT GUTENBERG EBOOK 1260,0,Jane Eyre
1,,1,Jane Eyre
2,JANE EYRE,2,Jane Eyre
3,AN AUTOBIOGRAPHY,3,Jane Eyre
4,,4,Jane Eyre


In [2]:
# We split the data into words
# We first split the text column into a list of words
bronte_books_df['word'] = bronte_books_df['line'].str.split()

# Explode the words column to create a new row for each word (this creates a separate row for each word from the newly created words list)
bronte_books_df = bronte_books_df.explode('word')

# Reset the index of the dataframe (we want to index each word now)
bronte_books_df = bronte_books_df.reset_index(drop=True)
bronte_books_df.head()

,line,line_number,book,word
0,START OF THE PROJECT GUTENBERG EBOOK 1260,0,Jane Eyre,START
1,START OF THE PROJECT GUTENBERG EBOOK 1260,0,Jane Eyre,OF
2,START OF THE PROJECT GUTENBERG EBOOK 1260,0,Jane Eyre,THE
3,START OF THE PROJECT GUTENBERG EBOOK 1260,0,Jane Eyre,PROJECT
4,START OF THE PROJECT GUTENBERG EBOOK 1260,0,Jane Eyre,GUTENBERG


In [3]:
# For our investigations the line & line_number columns will not be necessary, so we will remove them
bronte_books_df = bronte_books_df[['book', 'word']]
bronte_books_df

,book,word
0,Jane Eyre,START
1,Jane Eyre,OF
2,Jane Eyre,THE
3,Jane Eyre,PROJECT
4,Jane Eyre,GUTENBERG
...,...,...
575971,Agnes Grey,THE
575972,Agnes Grey,PROJECT
575973,Agnes Grey,GUTENBERG
575974,Agnes Grey,EBOOK


### Word counting revisited

In [4]:
# Let's count the occurrences of each word - this is a prerequisite for finding term frequency
count_df = bronte_books_df.groupby('word')['word'].count() # Group by word column, then only keep the word column and perform the counting

# Let's sort by term frequency
count_df_sorted = count_df.sort_values(ascending=False)

count_df_sorted.head(10)

word
the    21916
and    19421
I      18440
to     15556
of     13036
a      12126
in      7929
was     7438
you     6263
her     5981
Name: word, dtype: int64

In [5]:
# The .size() method operates similary, but differs slightly in output format
# .size() also counts null values, which .count() does not
bronte_books_df.groupby(['word']).size().sort_values(ascending=False).reset_index(name='count')

,word,count
0,the,21916
1,and,19421
2,I,18440
3,to,15556
4,of,13036
...,...,...
30097,yoursshe,1
30098,yoursthe,1
30099,youtell,1
30100,youreward,1


In [6]:
# Groupby allows grouping based on multiple columns
bronte_books_df.groupby(['word', 'book']).size().sort_values(ascending=False).reset_index(name='count')

,word,book,count
0,the,Vilette,7725
1,the,Jane Eyre,7332
2,I,Jane Eyre,7009
3,and,Jane Eyre,6263
4,and,Vilette,6097
...,...,...,...
52935,zigzag,Vilette,1
52936,zigzags,Vilette,1
52937,zle,Vilette,1
52938,hitting,Wuthering Heights,1


### Aggregate
One useful and elegant way of counting/aggregating data in pandas is by using the .agg() method.


In [7]:
# We group our data by words, then we aggregate and can decide what information we want to display for each column

# setting 'first' for the book column means that in the new dataframe we will display the first book on which each word occurs (in the book column)
# setting 'count' for the word column means that in the new dataframe we will display the count of given word (in the word column)
count_df = bronte_books_df.groupby('word').agg({'book': 'first', 'word': 'count'})
count_df

# Another way to describe the line above is - for each group (in our case a group = a word and all its appearances) we show on the 'book' column the first book from that group and on the 'word' column the total count of entries from that group
# .agg() is more flexible than .apply() and allows multi-column aggregations like the one we see above, each of which can be different - e.g. first and count

,book,word
word,,
07042,Wuthering Heights,1
1,Wuthering Heights,4
10,Vilette,1
1260,Jane Eyre,2
13th,Jane Eyre,1
...,...,...
zigzag,Jane Eyre,2
zigzags,Vilette,1
zle,Vilette,1


In [8]:
# Because we used groupby, the 'word' keyword has become both an index and a column name
# To get rid of any naming problems down the line, we will rename the column name 'word' to 'count'
count_df = count_df.rename(columns={'word': 'count'})

# Sorting values based on count column
count_df.sort_values('count', ascending=False)

,book,count
word,,
the,Jane Eyre,21916
and,Jane Eyre,19421
I,Jane Eyre,18440
to,Jane Eyre,15556
of,Jane Eyre,13036
...,...,...
yoursshe,Jane Eyre,1
yoursthe,Vilette,1
youtell,Agnes Grey,1


### Merging Dataframes

What we want next is to have a dataframe in which we know how many times each word appears per book and how many times it appears in all of the books.

It is sometimes very useful to merge together two dataframes and this is what we're going to do to get our desired dataframe.

In [9]:
count_df_1 = bronte_books_df.groupby(['word', 'book']).size().sort_values(ascending=False).reset_index(name='count') # How many appearances each word has in each book
count_df_1

,word,book,count
0,the,Vilette,7725
1,the,Jane Eyre,7332
2,I,Jane Eyre,7009
3,and,Jane Eyre,6263
4,and,Vilette,6097
...,...,...,...
52935,zigzag,Vilette,1
52936,zigzags,Vilette,1
52937,zle,Vilette,1
52938,hitting,Wuthering Heights,1


In [10]:
count_df_2 = bronte_books_df.groupby(['book']).size().sort_values(ascending=False).reset_index(name='count') # How many words each book has
count_df_2

,book,count
0,Vilette,196246
1,Jane Eyre,189694
2,Wuthering Heights,121120
3,Agnes Grey,68916


In [11]:
book_words = count_df_1.merge(count_df_2, on='book')
book_words.head(10)

,word,book,count_x,count_y
0,the,Vilette,7725,196246
1,the,Jane Eyre,7332,189694
2,I,Jane Eyre,7009,189694
3,and,Jane Eyre,6263,189694
4,and,Vilette,6097,196246
5,I,Vilette,5762,196246
6,to,Jane Eyre,5030,189694
7,of,Vilette,4813,196246
8,to,Vilette,4656,196246
9,and,Wuthering Heights,4502,121120


In [12]:
book_words = book_words.rename(columns={'count_x': 'word_appearances_in_book', 'count_y': 'book_total_word_count'}) # Give more meaningful names
book_words.head(10)

,word,book,word_appearances_in_book,book_total_word_count
0,the,Vilette,7725,196246
1,the,Jane Eyre,7332,189694
2,I,Jane Eyre,7009,189694
3,and,Jane Eyre,6263,189694
4,and,Vilette,6097,196246
5,I,Vilette,5762,196246
6,to,Jane Eyre,5030,189694
7,of,Vilette,4813,196246
8,to,Vilette,4656,196246
9,and,Wuthering Heights,4502,121120


### Exercise 1

1. Add a **tf** (term frequency) column to your book_words dataframe.
2. Add a new idf column to your dataframe
3. Add the final tf-idf column to your dataframe
4. Display your dataframe's words in descending order of their tf-idf.

Term frequency says how frequently a given word appears in a book. The formula for calculating it is
    
    term_frequency = word_appearances_in_book / book_total_word_count

Idf or inverse document frequency is computed as **idf = log(N / n)**

where


```
N is the total number of documents (books) in your dataset and n is the number of documents containing the word.
```



Once you have tf and idf, the tf-idf is obtained by simply multiplying the two.

Hint: For ex. 1.2 the pandas **transform** function could come in handy.

In [13]:
import numpy as np

# 1) Term frequency (tf)
book_words['tf'] = book_words['word_appearances_in_book'] / book_words['book_total_word_count']

# 2) Inverse document frequency (idf = log(N / n))
N = book_words['book'].nunique()
book_words['n_docs_with_word'] = book_words.groupby('word')['book'].transform('nunique')
book_words['idf'] = np.log(N / book_words['n_docs_with_word'])

# 3) tf-idf
book_words['tf_idf'] = book_words['tf'] * book_words['idf']

# 4) Display words in descending order of tf-idf
book_words.sort_values('tf_idf', ascending=False)[['word', 'book', 'tf', 'idf', 'tf_idf']].head(20)

,word,book,tf,idf,tf_idf
157,Heathcliff,Wuthering Heights,0.003410,1.386294,0.004727
200,Linton,Wuthering Heights,0.002807,1.386294,0.003892
211,Rochester,Jane Eyre,0.001645,1.386294,0.002280
203,Catherine,Wuthering Heights,0.002749,0.693147,0.001906
408,Hareton,Wuthering Heights,0.001354,1.386294,0.001877
807,Murray,Agnes Grey,0.001190,1.386294,0.001649
298,Dr,Vilette,0.001147,1.386294,0.001589
311,Bretton,Vilette,0.001101,1.386294,0.001526
334,Graham,Vilette,0.001009,1.386294,0.001399
956,Weston,Agnes Grey,0.001001,1.386294,0.001388


# Language Models

A language model is a statistical model that can be used to estimate the probability of a sequence of words in a language. It is trained on a corpus of text data, and learns to predict the likelihood of observing a given sequence of words based on the frequency and context of those words in the training data.

Language models can be used for a variety of natural language processing tasks, such as text generation, machine translation, speech recognition, and more.

In [14]:
import nltk
from nltk.corpus import brown
from nltk import FreqDist
nltk.download('brown')

# load the Brown corpus
corpus = brown.words()

[nltk_data] Downloading package brown to
[nltk_data]     C:\Users\alexc\AppData\Roaming\nltk_data...
[nltk_data]   Unzipping corpora\brown.zip.


In this example, we're using the Brown corpus from the nltk library, which is a collection of text samples from a wide range of genres, including news, fiction, and academic writing.

In [15]:
print(corpus[1100:1110]) # Print a sample of 10 words from the corpus

['voters', '.', 'Despite', 'the', 'warning', ',', 'there', 'was', 'a', 'unanimous']


In [16]:
# create a frequency distribution of the words in the corpus
freq_dist = FreqDist(corpus)

# calculate the total number of words in the corpus
total_words = len(corpus)

# calculate the probability of each word in the corpus
word_probs = {word: freq_dist[word] / total_words for word in freq_dist.keys()}
print(word_probs['high']) # Probability of the word 'high' to appear

0.0003970058353829513


### Naive sentence generation

We're going to create a naive function that generates sentences using our language model.

In [17]:
# generate a sentence using the language model
import random

def generate_sentence(word_length = 10):
    sentence = []
    while len(sentence) < word_length:
        word = random.choices(list(word_probs.keys()), list(word_probs.values()))[0]
        sentence.append(word)
    return " ".join(sentence)

In [18]:
print(generate_sentence())

pencil crowds which cabin as had modern as colonial to


The sentences generated are likely not going to sound very good, since the model is extremely naive.

All that is happening is that each word in the sentence gets semi-randomly generated with the likelihood of it being chosen depending on its frequency in the Brown corpus.

# N-grams

So far we’ve considered words as individual units, and considered the relationship to their frequency of occurrence. However, many interesting text analyses are based on the relationships between words.
One such relationship is given by n-grams.

N-grams are groups of n consecutive words that appear in a given text corpus.

Bigrams are groups of 2 consecutive words (e.g. she went, he ate, car crashed)

Trigrams are groups of 3 consecutive words (e.g. she went home, he ate a, the car crashed).

In [19]:
# Example of what bigrams look like
bigrams = list(nltk.bigrams(corpus))
bigrams[:10]

[('The', 'Fulton'),
 ('Fulton', 'County'),
 ('County', 'Grand'),
 ('Grand', 'Jury'),
 ('Jury', 'said'),
 ('said', 'Friday'),
 ('Friday', 'an'),
 ('an', 'investigation'),
 ('investigation', 'of'),
 ('of', "Atlanta's")]

In [20]:
# Example of what trigrams look like
trigrams = list(nltk.trigrams(corpus))
trigrams[5:20]

[('said', 'Friday', 'an'),
 ('Friday', 'an', 'investigation'),
 ('an', 'investigation', 'of'),
 ('investigation', 'of', "Atlanta's"),
 ('of', "Atlanta's", 'recent'),
 ("Atlanta's", 'recent', 'primary'),
 ('recent', 'primary', 'election'),
 ('primary', 'election', 'produced'),
 ('election', 'produced', '``'),
 ('produced', '``', 'no'),
 ('``', 'no', 'evidence'),
 ('no', 'evidence', "''"),
 ('evidence', "''", 'that'),
 ("''", 'that', 'any'),
 ('that', 'any', 'irregularities')]

### Naive next word prediction

Knowing that word relations are pretty important in our language, let's create a function that predicts what the next word in a sentence would be using a simple **bigram** language model.

In [21]:
from nltk.corpus import brown
import random

# get the words from the Brown corpus
corpus = brown.words()

# create bigrams from the corpus
bigrams = list(nltk.bigrams(corpus))

# calculate the frequency distribution of the bigrams
bigram_freqdist = nltk.FreqDist(bigrams)

# calculate the total number of bigrams in the corpus
total_bigrams = len(bigrams)

# create a function to generate the next word based on the previous word
def generate_next_word(sentence):
    prev_word = sentence.split()[-1]
    possible_words = {}
    for bigram in bigram_freqdist:
        if bigram[0] == prev_word:
            possible_words[bigram[1]] = bigram_freqdist[bigram] / total_bigrams
    if possible_words:
        return max(possible_words, key=possible_words.get)
    else:
        return None

In [22]:
# predict the next word for a given context
context = "The director"
next_word = generate_next_word(context)
print(f"The predicted next word for '{context}' is '{next_word}'")

The predicted next word for 'The director' is 'of'


### Exercise 2
1. Create a function that takes as input the number of words to generate and generates a sentence using the previous bigram language model. You can start with a random first word from the brown corpus and then use generate_next_word(sentence) function to help you.

2. Create a function that predicts the next word of a sentence by looking at the previous two words. This means you will create a trigram language model - use the same Brown corpus as before.

In [23]:
import random
import nltk

def generate_sentence_bigram(num_words):
    if num_words <= 0:
        return ""

    sentence_words = [random.choice(list(corpus))]  # random first word

    while len(sentence_words) < num_words:
        current_sentence = " ".join(sentence_words)
        next_word = generate_next_word(current_sentence) 

    

        sentence_words.append(next_word)

    return " ".join(sentence_words)


print("Bigram-generated sentence:")
print(generate_sentence_bigram(12))

trigram_freqdist = nltk.FreqDist(nltk.trigrams(corpus))

def predict_next_word_trigram(sentence):
    words = sentence.split()
    if len(words) < 2:
        return None

    prev_word1, prev_word2 = words[-2], words[-1]
    candidates = {}

    for (w1, w2, w3), count in trigram_freqdist.items():
        if w1 == prev_word1 and w2 == prev_word2:
            candidates[w3] = count

    if candidates:
        return max(candidates, key=candidates.get)

    return None


context = "The female"
print(f"Trigram next-word prediction for '{context}': {predict_next_word_trigram(context)}")

Bigram-generated sentence:
a few years ago , and the same time , and the
Trigram next-word prediction for 'The female': parasite


### N-grams in dataframes

Let's get back to our books.
We'll create a dataframe containing information about the bigrams in our books corpus.


In [24]:
# The simplest way to do this would be to create the dataframe directly from bigrams rather than unigrams (single words)
import nltk
nltk.download('punkt')
nltk.download('punkt_tab')

bronte1_bigrams = list(nltk.bigrams(nltk.word_tokenize(bronte1)))
bronte1_df = pd.DataFrame(bronte1_bigrams, columns=['Word 1', 'Word 2'])

bronte2_bigrams = list(nltk.bigrams(nltk.word_tokenize(bronte2)))
bronte2_df = pd.DataFrame(bronte2_bigrams, columns=['Word 1', 'Word 2'])

bronte3_bigrams = list(nltk.bigrams(nltk.word_tokenize(bronte3)))
bronte3_df = pd.DataFrame(bronte3_bigrams, columns=['Word 1', 'Word 2'])

bronte4_bigrams = list(nltk.bigrams(nltk.word_tokenize(bronte4)))
bronte4_df = pd.DataFrame(bronte4_bigrams, columns=['Word 1', 'Word 2'])


# We’ll want to know which content comes from which book
bronte1_df = bronte1_df.assign(book = 'Jane Eyre')
bronte2_df = bronte2_df.assign(book = 'Wuthering Heights')
bronte3_df = bronte3_df.assign(book = 'Vilette')
bronte4_df = bronte4_df.assign(book = 'Agnes Grey')

# Finally, we concatenate the books into one dataframe
books = [bronte1_df, bronte2_df, bronte3_df, bronte4_df]
bronte_books_df = pd.concat(books)
bronte_books_df.head()

[nltk_data] Downloading package punkt to
[nltk_data]     C:\Users\alexc\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package punkt_tab to
[nltk_data]     C:\Users\alexc\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!


,Word 1,Word 2,book
0,START,OF,Jane Eyre
1,OF,THE,Jane Eyre
2,THE,PROJECT,Jane Eyre
3,PROJECT,GUTENBERG,Jane Eyre
4,GUTENBERG,EBOOK,Jane Eyre


### Exercise 3

1. Add a **bigram** column that shows the entire bigrams ("The Project" and "Project Gutenberg" are examples of this column's values), not just the separate words.
2. Clean the dataframe by removing stop words.
3. Display the most frequently occuring 10 bigrams.

In [25]:
import nltk
from nltk.corpus import stopwords

nltk.download('stopwords')

# 1) Add full bigram text column
bronte_books_df['bigram'] = bronte_books_df['Word 1'] + ' ' + bronte_books_df['Word 2']

# 2) Remove rows where either word is a stop word
stop_words = set(stopwords.words('english'))
clean_bigrams_df = bronte_books_df[
    ~bronte_books_df['Word 1'].str.lower().isin(stop_words)
    & ~bronte_books_df['Word 2'].str.lower().isin(stop_words)
]

# 3) Display the 10 most frequent bigrams
top_10_bigrams = clean_bigrams_df['bigram'].value_counts().head(10)
print('Top 10 most frequent bigrams:')
print(top_10_bigrams)

[nltk_data] Downloading package stopwords to
[nltk_data]     C:\Users\alexc\AppData\Roaming\nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


Top 10 most frequent bigrams:
bigram
Mr Rochester         281
Dr John              126
St John              119
Mr Heathcliff        118
Mrs Fairfax          107
Madame Beck          101
Mrs Bretton           92
Project Gutenberg     83
young lady            74
Miss Grey             71
Name: count, dtype: int64


### Exercise 4

1. Create a dataframe containing the **bigram, word1, word2** and **book** columns for the following 4 books and remove stop words:
        https://www.gutenberg.org/cache/epub/1228/pg1228.txt - On the Origin of Species, by Charles Darwin

        https://www.gutenberg.org/cache/epub/4363/pg4363.txt - Beyond Good and Evil, by Friedrich Nietzsche

        https://www.gutenberg.org/cache/epub/3296/pg3296.txt - The Confessions of Saint Augustine, by Saint Augustine

        https://www.gutenberg.org/files/1661/1661-0.txt - The Adventures of Sherlock Holmes, by Arthur Conan Doyle

2. Display the most frequent 8 words of each book (use word1 column when counting)

3. Display the most relevant 8 words of each book based on tf-idf (use word1 column when counting)

4. Display the most relevant 5 bigrams of each book based on tf-idf

5. Display the most frequent 5 street names found in the entire 4 book corpus. The book they are coming from should also be visible.

6. Choose a fixed word1 of your choice and find the most common 5 bigrams in each book that have word1 equal to the word you chose.




In [1]:
import re
import requests
import pandas as pd
import nltk
from nltk.corpus import stopwords
from sklearn.feature_extraction.text import TfidfVectorizer

# NLTK resources
nltk.download('punkt')
nltk.download('punkt_tab')
nltk.download('stopwords')

book_sources = [
    ('https://www.gutenberg.org/cache/epub/1228/pg1228.txt', 'On the Origin of Species'),
    ('https://www.gutenberg.org/cache/epub/4363/pg4363.txt', 'Beyond Good and Evil'),
    ('https://www.gutenberg.org/cache/epub/3296/pg3296.txt', 'The Confessions of Saint Augustine'),
    ('https://www.gutenberg.org/files/1661/1661-0.txt', 'The Adventures of Sherlock Holmes')
]

stop_words = set(stopwords.words('english'))

def fetch_text(url):
    response = requests.get(url, timeout=60)
    response.raise_for_status()
    return response.text

def normalize_tokens(text):
    # Keep alphabetic tokens and apostrophes, lowercase for consistent counting.
    raw_tokens = nltk.word_tokenize(text)
    return [t.lower() for t in raw_tokens if re.fullmatch(r"[A-Za-z]+(?:'[A-Za-z]+)?", t)]

all_book_dfs = []
for url, title in book_sources:
    text = fetch_text(url)
    tokens = normalize_tokens(text)
    pairs = list(nltk.bigrams(tokens))

    df = pd.DataFrame(pairs, columns=['word1', 'word2'])
    df['book'] = title
    df['bigram'] = df['word1'] + ' ' + df['word2']

    # Remove stop words from either side of the bigram.
    df = df[~df['word1'].isin(stop_words) & ~df['word2'].isin(stop_words)].copy()
    all_book_dfs.append(df)

# 1) Combined dataframe with bigram, word1, word2, book
exercise4_df = pd.concat(all_book_dfs, ignore_index=True)
exercise4_df = exercise4_df[['bigram', 'word1', 'word2', 'book']]
print('1) Exercise 4 dataframe preview:')
display(exercise4_df.head())

# 2) Most frequent 8 words per book (count by word1)
freq8_words_per_book = (
    exercise4_df.groupby('book')['word1']
    .value_counts()
    .groupby(level=0)
    .head(8)
    .rename('count')
    .reset_index()
    .sort_values(['book', 'count'], ascending=[True, False])
)
print('2) Most frequent 8 words per book (using word1):')
display(freq8_words_per_book)

# Build per-book documents from word1 and bigram
book_order = [title for _, title in book_sources]
word_docs = []
bigram_docs = []
for title in book_order:
    sub = exercise4_df[exercise4_df['book'] == title]
    word_docs.append(' '.join(sub['word1'].tolist()))
    # Replace spaces in bigrams so each bigram is one token for vectorizer.
    bigram_docs.append(' '.join(sub['bigram'].str.replace(' ', '__', regex=False).tolist()))

# 3) Most relevant 8 words per book based on tf-idf (word1)
word_vectorizer = TfidfVectorizer()
word_tfidf = word_vectorizer.fit_transform(word_docs)
word_terms = word_vectorizer.get_feature_names_out()

top8_tfidf_words = []
for i, title in enumerate(book_order):
    row = word_tfidf.getrow(i).toarray().ravel()
    top_idx = row.argsort()[-8:][::-1]
    for idx in top_idx:
        top8_tfidf_words.append({
            'book': title,
            'word1': word_terms[idx],
            'tfidf': row[idx]
        })
top8_tfidf_words_df = pd.DataFrame(top8_tfidf_words)
print('3) Most relevant 8 words per book by tf-idf (using word1):')
display(top8_tfidf_words_df)

# 4) Most relevant 5 bigrams per book based on tf-idf
bigram_vectorizer = TfidfVectorizer()
bigram_tfidf = bigram_vectorizer.fit_transform(bigram_docs)
bigram_terms = bigram_vectorizer.get_feature_names_out()

top5_tfidf_bigrams = []
for i, title in enumerate(book_order):
    row = bigram_tfidf.getrow(i).toarray().ravel()
    top_idx = row.argsort()[-5:][::-1]
    for idx in top_idx:
        top5_tfidf_bigrams.append({
            'book': title,
            'bigram': bigram_terms[idx].replace('__', ' '),
            'tfidf': row[idx]
        })
top5_tfidf_bigrams_df = pd.DataFrame(top5_tfidf_bigrams)
print('4) Most relevant 5 bigrams per book by tf-idf:')
display(top5_tfidf_bigrams_df)

# 5) Most frequent 5 street names in full corpus with source book visible
street_suffixes = {'street', 'road', 'lane', 'avenue', 'court', 'place', 'square', 'way'}
street_df = exercise4_df[exercise4_df['word2'].isin(street_suffixes)].copy()
street_df['street_name'] = (street_df['word1'] + ' ' + street_df['word2']).str.title()

street_counts = (
    street_df.groupby(['street_name', 'book'])
    .size()
    .rename('count')
    .reset_index()
    .sort_values('count', ascending=False)
    .head(5)
)
print('5) Most frequent 5 street names in the corpus (with book):')
display(street_counts)

# 6) Fixed word1 and top 5 bigrams per book
fixed_word1 = 'man'
fixed_word_df = exercise4_df[exercise4_df['word1'] == fixed_word1].copy()
fixed_top5_per_book = (
    fixed_word_df.groupby('book')['bigram']
    .value_counts()
    .groupby(level=0)
    .head(5)
    .rename('count')
    .reset_index()
    .sort_values(['book', 'count'], ascending=[True, False])
)

print(f"6) Top 5 bigrams per book with fixed word1 = '{fixed_word1}':")
display(fixed_top5_per_book)

[nltk_data] Downloading package punkt to
[nltk_data]     C:\Users\alexc\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package punkt_tab to
[nltk_data]     C:\Users\alexc\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!
[nltk_data] Downloading package stopwords to
[nltk_data]     C:\Users\alexc\AppData\Roaming\nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


1) Exercise 4 dataframe preview:


,bigram,word1,word2,book
0,project gutenberg,project,gutenberg,On the Origin of Species
1,gutenberg ebook,gutenberg,ebook,On the Origin of Species
2,natural selection,natural,selection,On the Origin of Species
3,anyone anywhere,anyone,anywhere,On the Origin of Species
4,united states,united,states,On the Origin of Species


2) Most frequent 8 words per book (using word1):


,book,word1,count
0,Beyond Good and Evil,one,264
1,Beyond Good and Evil,every,135
2,Beyond Good and Evil,project,89
3,Beyond Good and Evil,new,84
4,Beyond Good and Evil,good,80
5,Beyond Good and Evil,still,78
6,Beyond Good and Evil,german,75
7,Beyond Good and Evil,man,75
8,On the Origin of Species,one,445
9,On the Origin of Species,natural,378


...

5) Most frequent 5 street names in the corpus (with book):


,street_name,book,count
4,Baker Street,The Adventures of Sherlock Holmes,28
116,Swandam Lane,The Adventures of Sherlock Holmes,8
125,Takes Place,The Confessions of Saint Augustine,7
31,Every Way,The Adventures of Sherlock Holmes,6
40,First Place,On the Origin of Species,6


6) Top 5 bigrams per book with fixed word1 = 'man':


,book,bigram,count
0,Beyond Good and Evil,man could,3
1,Beyond Good and Evil,man must,3
2,Beyond Good and Evil,man generally,2
3,Beyond Good and Evil,man without,2
4,Beyond Good and Evil,man would,2
5,On the Origin of Species,man selecting,2
6,On the Origin of Species,man wing,2
7,On the Origin of Species,man would,2
8,On the Origin of Species,man adds,1
9,On the Origin of Species,man appear,1


,bigram,word1,word2,book
0,project gutenberg,project,gutenberg,On the Origin of Species
1,gutenberg ebook,gutenberg,ebook,On the Origin of Species
2,natural selection,natural,selection,On the Origin of Species
3,anyone anywhere,anyone,anywhere,On the Origin of Species
4,united states,united,states,On the Origin of Species


# Exercise 5
1. Create a dataframe containing all the columns you will need to do a trigram-based analysis for **a book of your choice**. Suggested columns: trigram, word1, word2, word3, book and potentially tf, idf, tf-idf.

2. Display the top 10 words by tf-idf
3. Display the most frequent 5 trigrams for 2 target words of your choice.

 This means you will choose 2 separate words from the book (suggestion is to choose relevant words, e.g. main character names), keep one of the 3 words from the trigram fixed, and then display the most frequent trigrams co-occurring with your word. E.g. You choose 'John' as one of your two words - then you set it as a fixed word (up to you if it should be word1, word2 or word3), and find what are the most frequent trigrams that have John in that fixed position.

 Do this for two separate words, some suggestions are to use the protagonist and the antagonist of your story, or to use opposing principles/subjects from your work as your target words.




In [3]:
import re
import string
import requests
import numpy as np
import pandas as pd
import nltk
from collections import Counter
from nltk.corpus import stopwords

nltk.download('stopwords')


book_title = 'The Adventures of Sherlock Holmes'
book_url = 'https://www.gutenberg.org/files/1661/1661-0.txt'

response = requests.get(book_url)
book_text = response.text

allowed_chars = string.ascii_letters + string.digits + string.whitespace
book_text = ''.join(c for c in book_text if c in allowed_chars)

# Build line dataframe first, then explode to words
book_lines = book_text.splitlines()
book_df = pd.DataFrame({
    'line': book_lines,
    'line_number': list(range(len(book_lines)))
})
book_df['book'] = book_title
book_df['word'] = book_df['line'].str.split()

book_words_df = (
    book_df[['book', 'line_number', 'word']]
    .explode('word')
    .dropna()
    .reset_index(drop=True)
)

# Lowercase, keep alphabetic tokens, remove stop words
book_words_df['word'] = book_words_df['word'].str.lower()
book_words_df = book_words_df[book_words_df['word'].str.fullmatch(r'[a-z]+', na=False)]
stop_words = set(stopwords.words('english'))
book_words_df = book_words_df[~book_words_df['word'].isin(stop_words)].reset_index(drop=True)


all_tokens = book_words_df['word'].tolist()
trigram_tuples = list(nltk.trigrams(all_tokens))

trigram_df = pd.DataFrame(trigram_tuples, columns=['word1', 'word2', 'word3'])
trigram_df['trigram'] = trigram_df['word1'] + ' ' + trigram_df['word2'] + ' ' + trigram_df['word3']
trigram_df['book'] = book_title

trigram_counts = (
    trigram_df['trigram']
    .value_counts()
    .rename_axis('trigram')
    .reset_index(name='count')
)
trigram_stats = trigram_counts.copy()

# Use each original line as a document for trigram IDF
line_docs = (
    book_words_df.groupby('line_number')['word']
    .apply(list)
    .tolist()
)
line_docs = [doc for doc in line_docs if len(doc) >= 3]
N_docs = len(line_docs)

doc_freq_counter = Counter()
for doc in line_docs:
    doc_trigrams = set(' '.join(t) for t in nltk.trigrams(doc))
    doc_freq_counter.update(doc_trigrams)

trigram_stats['tf'] = trigram_stats['count'] / trigram_stats['count'].sum()
trigram_stats['n_docs_with_trigram'] = trigram_stats['trigram'].map(lambda x: doc_freq_counter.get(x, 0))
trigram_stats['idf'] = np.log(N_docs / trigram_stats['n_docs_with_trigram'].clip(lower=1))
trigram_stats['tf_idf'] = trigram_stats['tf'] * trigram_stats['idf']

exercise5_df = trigram_df.merge(
    trigram_stats[['trigram', 'count', 'tf', 'n_docs_with_trigram', 'idf', 'tf_idf']],
    on='trigram',
    how='left'
)
exercise5_df = exercise5_df[['trigram', 'word1', 'word2', 'word3', 'book', 'count', 'tf', 'n_docs_with_trigram', 'idf', 'tf_idf']]

print('1) Trigram dataframe preview (lab-style reading and cleaning):')
display(exercise5_df.head())

word_counts = Counter(all_tokens)
total_words = sum(word_counts.values())

word_doc_freq = Counter()
line_docs_words = (
    book_words_df.groupby('line_number')['word']
    .apply(list)
    .tolist()
)
line_docs_words = [doc for doc in line_docs_words if len(doc) > 0]
for doc in line_docs_words:
    word_doc_freq.update(set(doc))

word_stats = pd.DataFrame({'word': list(word_counts.keys())})
word_stats['count'] = word_stats['word'].map(word_counts)
word_stats['tf'] = word_stats['count'] / total_words
word_stats['n_docs_with_word'] = word_stats['word'].map(word_doc_freq)
word_stats['idf'] = np.log(len(line_docs_words) / word_stats['n_docs_with_word'].clip(lower=1))
word_stats['tf_idf'] = word_stats['tf'] * word_stats['idf']

top10_words = word_stats.sort_values('tf_idf', ascending=False).head(10)
print('2) Top 10 words by tf-idf:')
display(top10_words[['word', 'count', 'tf', 'idf', 'tf_idf']])


target_words = ['sherlock', 'watson']

for target in target_words:
    top5_target_trigrams = (
        exercise5_df[exercise5_df['word1'] == target]['trigram']
        .value_counts()
        .rename_axis('trigram')
        .reset_index(name='count')
        .head(5)
    )
    top5_target_trigrams['target_word'] = target
    top5_target_trigrams = top5_target_trigrams[['target_word', 'trigram', 'count']]

    print(f"3) Top 5 most frequent trigrams with fixed word1 = '{target}':")
    display(top5_target_trigrams)

[nltk_data] Downloading package stopwords to
[nltk_data]     C:\Users\alexc\AppData\Roaming\nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


1) Trigram dataframe preview (lab-style reading and cleaning):


,trigram,word1,word2,word3,book,count,tf,n_docs_with_trigram,idf,tf_idf
0,project gutenberg ebook,project,gutenberg,ebook,The Adventures of Sherlock Holmes,3,0.000062,3,7.936064,0.000493
1,gutenberg ebook adventures,gutenberg,ebook,adventures,The Adventures of Sherlock Holmes,3,0.000062,3,7.936064,0.000493
2,ebook adventures sherlock,ebook,adventures,sherlock,The Adventures of Sherlock Holmes,3,0.000062,3,7.936064,0.000493
3,adventures sherlock holmes,adventures,sherlock,holmes,The Adventures of Sherlock Holmes,5,0.000104,3,7.936064,0.000822
4,sherlock holmes arthur,sherlock,holmes,arthur,The Adventures of Sherlock Holmes,2,0.000041,0,9.034677,0.000374


2) Top 10 words by tf-idf:


,word,count,tf,idf,tf_idf
388,said,486,0.010065,2.982708,0.030020
160,upon,466,0.009650,3.040232,0.029340
5,holmes,462,0.009568,3.029326,0.028983
112,one,370,0.007662,3.282192,0.025149
127,would,327,0.006772,3.410152,0.023093
473,could,287,0.005944,3.535103,0.021011
72,man,288,0.005964,3.513750,0.020957
1239,mr,274,0.005674,3.571738,0.020267
177,little,269,0.005571,3.579228,0.019939
294,see,231,0.004784,3.737775,0.017881


3) Top 5 most frequent trigrams with fixed word1 = 'sherlock':


,target_word,trigram,count
0,sherlock,sherlock holmes sat,7
1,sherlock,sherlock holmes rather,2
2,sherlock,sherlock holmes arthur,2
3,sherlock,sherlock holmes said,2
4,sherlock,sherlock holmes laughing,2


3) Top 5 most frequent trigrams with fixed word1 = 'watson':


,target_word,trigram,count
0,watson,watson said holmes,6
1,watson,watson speak freely,2
2,watson,watson said makes,2
3,watson,watson nothing else,1
4,watson,watson practice observe,1
